# Shrink-Rotate tesselations

<p align="center">
  <img src="images/shrink-rotate/Curved Crossroads (4 to 6).jpg" style="height: 240px" />
  <img src="images/shrink-rotate/Seven Flowers.jpg" style="height: 240px" />
  <img src="images/shrink-rotate/7.4.3 Circles.jpg" style="height: 240px" />
</p>


This is one of the simplest styles of origami tesselations, both regarding the algorithm to construct them, as well as the number of creases per edge in the underlying tiling. Paired with the fact that many tilings work with the algorithm (the condition being that they posess a [reciprocal figure](https://en.wikipedia.org/wiki/Cremona_diagram)), they are your best bet to quickly generate a realistically foldable pattern from a bigger or particularily intricate tiling. 

#TODO: history

A mathematically rigurous treatment of shrink-rotate tesselations can be found in Robert J. Lang's [book on origami tesselations](https://langorigami.com/publication/twists-tilings-and-tessellations-mathematical-methods-for-geometric-origami/).


### The Algorithm

The algorithm can be described in three steps:

1. Compute a **reciprocal figure**: For each face of the input graph, find a point so that the segment connecting two such points across an edge is perpendicular to that edge. The position is stored as `face['reciprocal_pos']`.
2. **Shrink-rotate** every face: Shrink each face by a given factor, and rotate around its reciprocal point. The default scale factor is 0.5 and the rotation is $\\pi/5$, but each is a free parameter.
3. **Connect** corners of these faces which were touching in the original tiling (before being shrunk) by creases. 

However, the implementation actually performs step (3) before step (2), via the ``shrink_rotate`` conway operator.

In [ ]:
%matplotlib widget
import pleat as ec
from pleat.example_graphs import from_tiles
from pleat.example_tilesets import t_4_6_12
from pleat.shrink_rotate import (
    shrink_rotate_pattern,
    crease_orientation
)
from pleat.shrink_rotate import ShrinkRotateExplorer
from pleat.rendering import CREASE_PATTERN_PRESET

In [ ]:
# Build a tiling, assign face z-order (controls mountain/valley), and build the SRG.
G = from_tiles(t_4_6_12(), rings=3)
crease_orientation.assign_this_way_by_face_degree(G)

SRG = shrink_rotate_pattern(G, simplify_boundary=True)

SRG.show(**CREASE_PATTERN_PRESET)

### Interacitve explorer

Inspired by the [Tess software](https://www.papermosaics.co.uk/software.html) by Alex Bateman, 
`pleat.shrink_rotate.ShrinkRotateExplorer` packages an interactive widget to tune the parameters of a shrink-rotate tesselation: Drag the sliders to vary the shrink factor and rotation angle and see the crease pattern update live. Toggle *folded* to preview a backlit version of the folded form, or *reparametrized* to switch to the equivalent (β, γ) parametrisation, which is typically more convenient for fine adjustments of the model.
The widget requires a Jupyter kernel with `%matplotlib widget`.

In [ ]:
explorer = ShrinkRotateExplorer(SRG)
explorer.display()

The widget mutates `v['pos']` on the SRG in place at every change, so downstream operations like `pleat.overlap.fold_complete(SRG)` always operate on the currently-displayed geometry.

In [ ]:
SRG.recompute_lengths_and_angles()
results = ec.overlap.fold_complete(SRG.copy(), quiet=True, overlap_eps=1e-6)
results.show()

### Crease Assignment

Every edge in the original graph corresponds to a parallelogram in the CP. For each such parallelogram, there are two valid mountain / valley crease assignments (two sides are mountains, two sides are valleys, the edges coming together at obtuse angles are the same type). 
<!-- TODO: insert picture showing each case -->
This can be chosen independently for each parallelogram, and is controlled by setting the ``THIS_WAY`` property of halfedges in the original tiling. This can be done conveniently with the different functions in ``pleat.shrink_rotate.crease_assignment``, before constructing the crease pattern.

In [ ]:
# central faces on top of outer ones

G = from_tiles(t_4_6_12(), rings=3)
crease_orientation.assign_this_way_from_center(G)
SRG = shrink_rotate_pattern(G, simplify_boundary=True)
ec.overlap.fold_complete(SRG, quiet=True, overlap_eps=1e-6).show()

In [ ]:
# no assigment -> random valid assignment found by layer ordering solver

G = from_tiles(t_4_6_12(), rings=3)
SRG = shrink_rotate_pattern(G, simplify_boundary=True)
ec.overlap.fold_complete(SRG, quiet=True, overlap_eps=1e-6).show()

In [ ]:
# hexagons on top of squares on top of dodecagons:

G = from_tiles(t_4_6_12(), rings=3)
for f in G.faces:
    if f.order() == 6:
        f['z_order'] = 2
    elif f.order() == 4:
        f['z_order'] = 1    
    else:
        f['z_order'] = 0
crease_orientation.assign_this_way_by_face_z_order(G)
SRG = shrink_rotate_pattern(G, simplify_boundary=True)
ec.overlap.fold_complete(SRG, quiet=True, overlap_eps=1e-6).show()